# Plant Disease Detection - MobileNetV2 Transfer Learning

In [ ]:
# Upload your kaggle.json file
from google.colab import files
files.upload()

In [ ]:
# Install required packages
!pip install -q kaggle "tensorflow>=2.17.0" gradio matplotlib

# Python imports and configuration
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# If dataset not downloaded yet, download and unzip
if not os.path.exists("/content/new-plant-diseases-dataset"):
    !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content
    !unzip -q /content/new-plant-diseases-dataset.zip -d /content/new-plant-diseases-dataset

# Confirm dataset presence
!ls -lah /content/new-plant-diseases-dataset | sed -n '1,20p'

In [ ]:
# User / experiment configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_EPOCHS = 8
FINETUNE_EPOCHS = 10
UNFREEZE_LAYERS = 40
SEED = 1337

# Dataset paths
train_dir = "/content/new-plant-diseases-dataset/New Plant Diseases Dataset (Augmented)/train"
valid_dir = "/content/new-plant-diseases-dataset/New Plant Diseases Dataset (Augmented)/valid"

for p in [train_dir, valid_dir]:
    if not os.path.exists(p):
        print(f"ERROR: path not found: {p}")
        sys.exit(1)

print("train_dir:", train_dir)
print("valid_dir:", valid_dir)

In [ ]:
import random
from IPython.display import Image, display

def show_sample_images(base_path, num_classes=5, images_per_class=3):
    """Display random sample images from the dataset"""
    class_names = os.listdir(base_path)
    random_classes = random.sample(class_names, min(num_classes, len(class_names)))
    fig, axes = plt.subplots(num_classes, images_per_class, figsize=(12, 3 * num_classes))
    fig.suptitle("Sample Images from Dataset", fontsize=16)
    for i, class_name in enumerate(random_classes):
        class_path = os.path.join(base_path, class_name)
        images = os.listdir(class_path)
        random_images = random.sample(images, min(images_per_class, len(images)))
        for j, img_name in enumerate(random_images):
            img_path = os.path.join(class_path, img_name)
            img = plt.imread(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis("off")
            if j == 0:
                axes[i, j].set_title(class_name.replace("___", "\n"), fontsize=10)
    plt.tight_layout()
    plt.show()

show_sample_images(train_dir)

In [ ]:
# Count images in each split
def count_images(directory):
    """Count total images and images per class"""
    total_images = 0
    class_counts = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            num_images = len(os.listdir(class_path))
            class_counts[class_name] = num_images
            total_images += num_images
    return total_images, class_counts

train_total, train_counts = count_images(train_dir)
valid_total, valid_counts = count_images(valid_dir)
print(f"Training images: {train_total}")
print(f"Validation images: {valid_total}")
print(f"Number of classes: {len(train_counts)}")
print("\nClass distribution (first 10):")
for class_name, count in list(train_counts.items())[:10]:
    print(f"{class_name}: {count} images")

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input

# Training data generator with stronger augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=22,
    zoom_range=0.2,
    width_shift_range=0.12,
    height_shift_range=0.12,
    shear_range=0.1,
    brightness_range=(0.85, 1.15),
    fill_mode="nearest"
 )

# Validation data generator (no augmentation)
valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

valid_gen = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
 )

print("EfficientNetB0 loaded successfully!")
print(f"Total layers in base model: {len(base_model.layers)}")

for layer in base_model.layers:
    layer.trainable = False

trainable_count = sum([tf.size(w).numpy() for w in base_model.trainable_weights])
non_trainable_count = sum([tf.size(w).numpy() for w in base_model.non_trainable_weights])
print(f"Trainable parameters: {trainable_count:,}")
print(f"Non-trainable parameters: {non_trainable_count:,}")
print("Backbone frozen for Phase 1 training")

In [ ]:
NUM_CLASSES = train_gen.num_classes

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.45)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="efficientnetb0_plant_disease_classifier")
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")]
)

print("Phase 1 compile complete")
print("Optimizer: Adam (lr=3e-4)")
print("Loss function: Categorical Crossentropy")
print("Metrics: Accuracy, Top-3 Accuracy")

In [ ]:
os.makedirs("/content/model_artifacts", exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "/content/model_artifacts/efficientnet_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger("/content/model_artifacts/training_log.csv"),
]

print("Callbacks configured:")
print("1. ModelCheckpoint - Saves best EfficientNet model")
print("2. ReduceLROnPlateau - Adjusts learning rate")
print("3. EarlyStopping - Prevents overfitting")
print("4. CSVLogger - Saves epoch-level metrics")

In [ ]:
print("Phase 1: Train classification head")
history_phase1 = model.fit(
    train_gen,
    epochs=INITIAL_EPOCHS,
    validation_data=valid_gen,
    callbacks=callbacks
)

In [ ]:
print("Phase 2: Fine-tuning top backbone layers")

base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")]
)

history_phase2 = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=INITIAL_EPOCHS + FINETUNE_EPOCHS,
    initial_epoch=history_phase1.epoch[-1] + 1,
    callbacks=callbacks
)

history_all = {}
for key in history_phase1.history.keys():
    history_all[key] = history_phase1.history[key] + history_phase2.history.get(key, [])

print(f"Fine-tuned top {UNFREEZE_LAYERS} layers")

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_all["accuracy"], label="train_accuracy")
plt.plot(history_all["val_accuracy"], label="val_accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history_all["loss"], label="train_loss")
plt.plot(history_all["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("Loss")
plt.show()

In [ ]:
val_metrics = model.evaluate(valid_gen, verbose=1)
for metric_name, metric_value in zip(model.metrics_names, val_metrics):
    print(f"Validation {metric_name}: {metric_value:.4f}")

In [ ]:
import json

final_path = "/content/model_artifacts/efficientnet_final.keras"
model.save(final_path)

class_names = [None] * len(train_gen.class_indices)
for class_name, class_id in train_gen.class_indices.items():
    class_names[class_id] = class_name

with open("/content/model_artifacts/class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

print("Saved final model to:", final_path)
print("Saved class names to: /content/model_artifacts/class_names.json")

In [ ]:
# Optional: zip and download all model artifacts from Colab
import shutil
from google.colab import files

archive_base = "/content/model_artifacts"
archive_path = shutil.make_archive(archive_base, "zip", "/content/model_artifacts")
print("Created:", archive_path)
files.download(archive_path)